
## Objective
Build and compare ARIMA and LSTM models to forecast Tesla (TSLA) stock prices.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pmdarima import auto_arima
from statsmodels.tsa.stattools import adfuller

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense


In [2]:
df_clean = pd.read_csv("../data/processed/clean_prices.csv", parse_dates=["Date"])


In [39]:
# Select Tesla data and clean NaNs BEFORE modeling
tsla_df = (
    df_clean[df_clean["Ticker"] == "TSLA"]
    .sort_values("Date")
    .reset_index(drop=True)
)

# Drop rows where Adj Close is NaN
tsla_df = tsla_df.dropna(subset=["Adj Close"])

tsla_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2775 entries, 0 to 2774
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    2775 non-null   datetime64[us]
 1   Open                    2775 non-null   float64       
 2   High                    2775 non-null   float64       
 3   Low                     2775 non-null   float64       
 4   Close                   2775 non-null   float64       
 5   Adj Close               2775 non-null   float64       
 6   Volume                  2775 non-null   float64       
 7   Ticker                  2775 non-null   str           
 8   Daily_Return            2774 non-null   float64       
 9   Rolling_30D_Volatility  2745 non-null   float64       
dtypes: datetime64[us](1), float64(8), str(1)
memory usage: 216.9 KB


PART A: ARIMA MODEL

In [40]:
train_df = tsla_df[tsla_df["Date"] < "2025-01-01"]
test_df  = tsla_df[tsla_df["Date"] >= "2025-01-01"]

y_train = train_df["Adj Close"].values
y_test  = test_df["Adj Close"].values

print("NaNs in y_train:", np.isnan(y_train).sum())
print("NaNs in y_test :", np.isnan(y_test).sum())


NaNs in y_train: 0
NaNs in y_test : 0


In [41]:
from pmdarima import auto_arima

arima_model = auto_arima(
    y_train,
    seasonal=False,
    trace=False,
    error_action="ignore",
    suppress_warnings=True,
    stepwise=True
)

arima_forecast = arima_model.predict(n_periods=len(y_test))


In [30]:
# Fit ARIMA model on training data
arima_model.fit(y_train)


,order,"(0, ...)"
,seasonal_order,"(0, ...)"
,start_params,None
,method,'lbfgs'
,maxiter,50
,suppress_warnings,True
,out_of_sample_size,0
,scoring,'mse'
,scoring_args,{}
,trend,None
,with_intercept,False


🔵 PART B: LSTM MODEL

In [31]:
# LSTM requires scaled data
scaler = MinMaxScaler()

scaled_prices = scaler.fit_transform(
    tsla_df[["Adj Close"]]
)


In [32]:
def create_sequences(data, window_size=60):
    """
    Create input-output sequences for LSTM.
    Uses previous 'window_size' days to predict the next day.
    """
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_prices, window_size=60)


In [33]:
train_size = len(train_df)

# Split sequences consistently with ARIMA split
X_train = X[:train_size-60]
X_test  = X[train_size-60:]

y_train_lstm = y[:train_size-60]
y_test_lstm  = y[train_size-60:]

# Reshape for LSTM: [samples, timesteps, features]
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)


In [34]:
lstm_model = Sequential([
    LSTM(50, input_shape=(60, 1)),
    Dense(1)
])

lstm_model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

lstm_model.summary()


d:\KAIM\weak-9\portfolio-optimization\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 50)             │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,451 (40.82 KB)

 Trainable params: 10,451 (40.82 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
lstm_model.fit(
    X_train,
    y_train_lstm,
    epochs=10,
    batch_size=32,
    verbose=1
)


Epoch 1/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0103
Epoch 2/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 7.8314e-04
Epoch 3/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 6.9525e-04
Epoch 4/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 6.2170e-04
Epoch 5/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 5.7110e-04
Epoch 6/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 5.0078e-04
Epoch 7/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 4.8093e-04
Epoch 8/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 4.3343e-04
Epoch 9/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 4.2712e-04
Epoch 10/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 4.1855e-04


In [36]:
# Generate LSTM predictions
lstm_predictions = lstm_model.predict(X_test)

# Inverse scale predictions and true values
lstm_predictions = scaler.inverse_transform(lstm_predictions)
y_test_actual = scaler.inverse_transform(
    y_test_lstm.reshape(-1, 1)
)


9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


🔵 PART C: MODEL EVALUATION

In [37]:
def evaluate_model(y_true, y_pred):
    """
    Compute MAE, RMSE, and MAPE.
    Uses NumPy arrays to avoid index-related issues.
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mae, rmse, mape


In [42]:
arima_mae, arima_rmse, arima_mape = evaluate_model(
    y_test,
    arima_forecast
)

arima_mae, arima_rmse, arima_mape


(69.4989569233191,
 np.float64(82.93461282588196),
 np.float64(22.557915255182202))

In [43]:
lstm_mae, lstm_rmse, lstm_mape = evaluate_model(
    y_test_actual.flatten(),
    lstm_predictions.flatten()
)

lstm_mae, lstm_rmse, lstm_mape


(17.86206614372814,
 np.float64(22.0883437583124),
 np.float64(4.974826765042677))

In [44]:
results = pd.DataFrame({
    "Model": ["ARIMA", "LSTM"],
    "MAE": [arima_mae, lstm_mae],
    "RMSE": [arima_rmse, lstm_rmse],
    "MAPE (%)": [arima_mape, lstm_mape]
})

results


,Model,MAE,RMSE,MAPE (%)
0,ARIMA,69.498957,82.934613,22.557915
1,LSTM,17.862066,22.088344,4.974827


## Model Comparison Discussion

ARIMA provides a strong and interpretable baseline model, performing well
with relatively low complexity. The LSTM model is capable of capturing
non-linear patterns in the data but requires more tuning and computational
resources.

The choice between models depends on the trade-off between interpretability
(ARIMA) and modeling flexibility (LSTM).
